In [1]:
# 1. load + feature matrix
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.read_parquet("../data/constructs1/slope_feature_table.parquet")
FEATS = ["elev","slope","aspect_sin","aspect_cos","roughness",
         "tri","tpi","ndvi","lst","dist"]
X = df[FEATS].to_numpy(np.float64)
print("rows:", X.shape[0], "| features:", X.shape[1],
      "| NaNs:", int(np.isnan(X).sum()))

rows: 2575353 | features: 10 | NaNs: 0


In [2]:
# 2 -- standardize and PCA (over all 6 slopes)
Z = StandardScaler().fit_transform(X)
pca = PCA().fit(Z)

evr = pca.explained_variance_ratio_
print("explained variance ratio:")
for i, e in enumerate(evr, 1):
    print(f"  PC{i}: {e*100:5.1f}% cum. {evr[:i].sum()*100:5.1f}%")

explained variance ratio:
  PC1:  29.1% cum.  29.1%
  PC2:  22.6% cum.  51.7%
  PC3:  15.1% cum.  66.9%
  PC4:  10.1% cum.  76.9%
  PC5:  10.0% cum.  86.9%
  PC6:   6.5% cum.  93.5%
  PC7:   3.0% cum.  96.4%
  PC8:   1.8% cum.  98.3%
  PC9:   1.3% cum.  99.6%
  PC10:   0.4% cum. 100.0%


In [3]:
# 3. -- loadings and feature contribution

load = pd.DataFrame(pca.components_.T, index = FEATS,
                    columns=[f"PC{i}" for i in range(1,11)])
print(load.round(2).iloc[:,:6]) # first 6 PCs, ~ 93.5% of variance

k = 6
contrib = (load.iloc[:, :k]**2 * evr[:k]).sum(axis=1)
contrib = (contrib / contrib.sum() * 100).sort_values(ascending=False)
print("\nfeature share of the variance captured by PC1-6 (%)")
print(contrib.round(1))

             PC1   PC2   PC3   PC4   PC5   PC6
elev        0.06  0.62 -0.08  0.05 -0.00  0.11
slope       0.55 -0.03 -0.14  0.00  0.01 -0.02
aspect_sin  0.06 -0.04  0.14  0.91 -0.29  0.25
aspect_cos  0.19 -0.06  0.69 -0.15  0.07  0.13
roughness   0.57 -0.02 -0.12 -0.02  0.01 -0.04
tri         0.56 -0.02 -0.11 -0.02 -0.01 -0.04
tpi        -0.00  0.00 -0.01  0.31  0.95 -0.01
ndvi        0.02 -0.50 -0.02 -0.18  0.07  0.74
lst        -0.11 -0.32 -0.62  0.09 -0.03 -0.05
dist       -0.02  0.51 -0.25 -0.12  0.04  0.59

feature share of the variance captured by PC1-6 (%)
tpi           10.7
aspect_sin    10.7
roughness     10.3
ndvi          10.3
tri           10.1
dist          10.0
slope          9.8
elev           9.5
aspect_cos     9.3
lst            9.3
dtype: float64


In [9]:
# 4. PCA over subsets
def pca_report(mask, label, k=6):
    Xs = df.loc[mask, FEATS].to_numpy(np.float64)
    Zs = StandardScaler().fit_transform(Xs)
    p = PCA().fit(Zs)
    L = p.components_.T.copy()
    for j in range(L.shape[1]):   # stabilize sign: max - |loading| positive
        if L[np.argmax(np.abs(L[:,j])), j] < 0:
            L[:,j] *= -1
    evr = p.explained_variance_ratio_
    load = pd.DataFrame(L[:,:k], index=FEATS, columns=[f"PC{i}" for i in range(1,k+1)])
    contrib = (load**2 * evr[:k]).sum(1); contrib = (contrib/contrib.sum() * 100)
    print(f"\n==== {label} (n={int(mask.sum()):,}) =====")
    #print("var%: " + " " .join(f"PC{i+1} {e*100:.1f}" for i, e in enumerate(evr[:k])) + f"  cum {evr[:k].sum()*100:.1f}")
    print(load.round(2))
    #print("contrib%:", {kk: round(vv,1) for kk, vv in contrib.items()})
    return p, load
    

In [10]:
reports = {}
for lab, m in [("NORTH_ALL", df.side=="N"), ("SOUTH_ALL",df.side=="S"),
               ("CANWELL", df.glacier=="canwell"), ("CASTNER", df.glacier=="castner"),
               ("FELS", df.glacier=="fels"), ("GLACIERS_ALL",pd.Series(True, index=df.index))]:
    reports[lab] = pca_report(m, lab)


==== NORTH_ALL (n=1,666,750) =====
             PC1   PC2   PC3   PC4   PC5   PC6
elev       -0.22  0.52 -0.15  0.02 -0.08  0.09
slope       0.51  0.25 -0.02  0.01 -0.06  0.01
aspect_sin  0.08 -0.02 -0.59 -0.07  0.76  0.23
aspect_cos -0.03  0.11  0.75  0.01  0.50  0.28
roughness   0.51  0.27  0.05  0.01 -0.03 -0.01
tri         0.51  0.27  0.04 -0.01 -0.01 -0.00
tpi        -0.00 -0.00 -0.04  1.00  0.07 -0.02
ndvi        0.20 -0.44  0.08  0.02 -0.13  0.64
lst         0.26 -0.39 -0.17  0.03 -0.23  0.15
dist       -0.23  0.40 -0.14  0.02 -0.30  0.66

==== SOUTH_ALL (n=908,603) =====
             PC1   PC2   PC3   PC4   PC5   PC6
elev        0.31  0.52 -0.03  0.12 -0.04  0.05
slope       0.50 -0.22  0.02 -0.08  0.05 -0.04
aspect_sin  0.02  0.04  0.79  0.02 -0.00  0.58
aspect_cos  0.09 -0.25  0.36  0.64 -0.24 -0.54
roughness   0.52 -0.23 -0.00 -0.17  0.07  0.01
tri         0.51 -0.22  0.00 -0.18  0.06  0.01
tpi        -0.00  0.01 -0.00  0.36  0.93 -0.00
ndvi       -0.03 -0.43 -0.28  0.45 -0